In [1]:
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "distilroberta-base"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
subset_size = 500
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 128 if device == "mps" else 64
max_length = 128
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "subset_size": subset_size,
    "device": device,
    "batch_size": batch_size,
    "max_length": max_length,
    "seed": seed,
})


{'model_name': 'distilroberta-base', 'dataset': 'glue/stsb', 'split': 'validation', 'subset_size': 500, 'device': 'mps', 'batch_size': 128, 'max_length': 128, 'seed': 42}


In [2]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
ds = ds.select(range(min(subset_size, len(ds))))
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())


{'num_examples': 500, 'columns': ['sentence1', 'sentence2', 'label']}
                              sentence1  \
0     A man with a hard hat is dancing.   
1      A young child is riding a horse.   
2  A man is feeding a mouse to a snake.   
3        A woman is playing the guitar.   
4         A woman is playing the flute.   

                                  sentence2  label  
0      A man wearing a hard hat is dancing.   5.00  
1                A child is riding a horse.   4.75  
2  The man is feeding a mouse to the snake.   5.00  
3                  A man is playing guitar.   2.40  
4                 A man is playing a flute.   2.75  


In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()
print(model_name)


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


distilroberta-base


In [4]:
def cls_pool(last_hidden_state):
    return last_hidden_state[:, 0, :]

def encode_texts(texts, batch_size=64, max_length=128):
    all_embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            encoded = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            encoded = {k: v.to(device) for k, v in encoded.items()}
            outputs = model(**encoded)
            pooled = cls_pool(outputs.last_hidden_state)
            pooled = F.normalize(pooled, p=2, dim=1)
            all_embeddings.append(pooled.cpu())
    return torch.cat(all_embeddings, dim=0).numpy()


In [5]:
sentences1 = df["sentence1"].tolist()
sentences2 = df["sentence2"].tolist()
labels = df["label"].to_numpy(dtype=np.float32)

emb1 = encode_texts(sentences1, batch_size=batch_size, max_length=max_length)
emb2 = encode_texts(sentences2, batch_size=batch_size, max_length=max_length)

cosine_similarity = np.sum(emb1 * emb2, axis=1)
predicted_score_0_5 = 2.5 * (cosine_similarity + 1.0)
absolute_error = np.abs(predicted_score_0_5 - labels)

norms1 = np.linalg.norm(emb1, axis=1)
norms2 = np.linalg.norm(emb2, axis=1)


In [6]:
pearson_corr = pearsonr(predicted_score_0_5, labels).statistic
spearman_corr = spearmanr(predicted_score_0_5, labels).statistic

results_df = df.copy()
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["absolute_error"] = absolute_error
results_df["embedding_norm_s1"] = norms1
results_df["embedding_norm_s2"] = norms2

print(results_df[["sentence1", "sentence2", "label", "cosine_similarity", "predicted_score_0_5", "absolute_error", "embedding_norm_s1", "embedding_norm_s2"]].head(10))


                              sentence1  \
0     A man with a hard hat is dancing.   
1      A young child is riding a horse.   
2  A man is feeding a mouse to a snake.   
3        A woman is playing the guitar.   
4         A woman is playing the flute.   
5          A woman is cutting an onion.   
6       A man is erasing a chalk board.   
7            A woman is carrying a boy.   
8        Three men are playing guitars.   
9               A woman peels a potato.   

                                  sentence2  label  cosine_similarity  \
0      A man wearing a hard hat is dancing.  5.000           0.999952   
1                A child is riding a horse.  4.750           0.999951   
2  The man is feeding a mouse to the snake.  5.000           0.999874   
3                  A man is playing guitar.  2.400           0.999714   
4                 A man is playing a flute.  2.750           0.999685   
5                  A man is cutting onions.  2.615           0.999697   
6       The man

In [7]:
runtime_seconds = time.time() - start_time

cosine_summary = pd.Series(cosine_similarity, name="cosine_similarity").agg(["mean", "std", "min", "max", "median"])
absolute_error_summary = pd.Series(absolute_error, name="absolute_error").agg(["mean", "std", "min", "max", "median"])
norm_summary = pd.DataFrame({
    "sentence1_norm": norms1,
    "sentence2_norm": norms2,
}).agg(["mean", "std", "min", "max"])

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {len(df)}")
print(f"pearson_correlation: {pearson_corr:.6f}")
print(f"spearman_correlation: {spearman_corr:.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")
print("raw_cosine_similarity_summary:")
print(cosine_summary)
print("absolute_error_summary:")
print(absolute_error_summary)
print("embedding_norm_summary:")
print(norm_summary)


device_used: mps
model_name: distilroberta-base
dataset_split: glue/stsb/validation
num_examples: 500
pearson_correlation: 0.478042
spearman_correlation: 0.676387
runtime_seconds: 326.28
raw_cosine_similarity_summary:
mean      0.999460
std       0.000464
min       0.996641
max       0.999972
median    0.999615
Name: cosine_similarity, dtype: float32
absolute_error_summary:
mean      2.611126
std       1.626917
min       0.000069
max       4.998567
median    2.498484
Name: absolute_error, dtype: float32
embedding_norm_summary:
      sentence1_norm  sentence2_norm
mean    1.000000e+00    1.000000e+00
std     4.563460e-08    5.103307e-08
min     9.999998e-01    9.999998e-01
max     1.000000e+00    1.000000e+00
